In [39]:
from pathlib import Path
import subprocess
import pandas as pd

In [40]:
PROJECT_ROOT = Path.cwd().parent

DATASET_PATH = PROJECT_ROOT / "data" / "raw" / "contrib-versions"

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATASET_PATH)
print("Dataset exists:", DATASET_PATH.exists())

Project root: c:\Users\arjun\Desktop\SneakyToS
Dataset path: c:\Users\arjun\Desktop\SneakyToS\data\raw\contrib-versions
Dataset exists: True


In [41]:
companies = [
    folder for folder in DATASET_PATH.iterdir()
    if folder.is_dir()
]

print("Number of company folders:", len(companies))

print("\nFirst 20 companies:")

for company in companies[:20]:
    print("-", company.name)

Number of company folders: 387

First 20 companies:
- .git
- 123Greetings
- Academia.edu
- ACDelco
- AcuityAds
- AdMob
- AdRoll
- AdSense
- AdSupply
- Aetna
- Airbnb
- Alexa
- Alibaba.com
- Allrecipes
- Allstate
- Alpha Vantage
- Amazon Advertising
- Amazon Appstore
- Amazon Coins
- Amazon Device


In [42]:
privacy_policies = []

for company in companies:
    policy_file = company / "Privacy Policy.md"

    if policy_file.exists():
        privacy_policies.append({
            "company": company.name,
            "path": policy_file
        })

privacy_df = pd.DataFrame(privacy_policies)

print("Companies with Privacy Policies:", len(privacy_df))

privacy_df.head(10)

Companies with Privacy Policies: 307


,company,path
0,123Greetings,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
1,Academia.edu,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
2,ACDelco,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
3,AcuityAds,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
4,AdRoll,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
5,AdSense,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
6,AdSupply,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
7,Aetna,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
8,Airbnb,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
9,Alibaba.com,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...


In [43]:
def get_git_history(repo_path, file_path):
    command = [
        "git",
        "-C",
        str(repo_path),
        "log",
        "--follow",
        "--format=%H|%aI|%s",
        "--",
        str(file_path)
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=True,
        encoding="utf-8",
        errors="replace"
    )

    return result.stdout

In [44]:
zoom_policy = DATASET_PATH / "Zoom" / "Privacy Policy.md"

zoom_history = get_git_history(
    DATASET_PATH,
    Path("Zoom") / "Privacy Policy.md"
)

print(zoom_history[:2000])

2a55363890072a783635c05ff3df6af901fcb1ef|2024-08-05T18:19:53Z|Record new changes of Zoom Privacy Policy
5f21c439d209a348c08bdf631cea3313c72f39ce|2024-07-19T18:19:37Z|Record new changes of Zoom Privacy Policy
5974f03468c0d63146c8a5f9755128a98a9daf71|2024-03-01T06:47:40Z|Apply technical or declaration upgrade on Zoom Privacy Policy
da732810c04d446cefbdeb27d82432307224d6a1|2024-03-17T06:48:06Z|Record new changes of Zoom Privacy Policy
d19d78ea80fa38c2fd34b6028e07f7da5f4e6073|2023-12-20T00:46:10Z|Record new changes of Zoom Privacy Policy
1559d97e5d0b44f00609200e65e1965f0bbb0fb9|2023-12-15T18:28:35Z|Record new changes of Zoom Privacy Policy
da1e3f2173580ff939692c2a75666b1d171c46ea|2023-12-02T11:18:10Z|Record new changes of Zoom Privacy Policy
05d6d4fe3ebb303de242cfcccbc20c9daec2b7cf|2023-09-08T21:57:39Z|Record new changes of Zoom Privacy Policy
1105702377e68c8806ba0eb9a0ae5b6628fde209|2023-09-05T12:18:54Z|Record new changes of Zoom Privacy Policy
87a1e21aab39dba80d29fa06da181c773805ab84|202

In [45]:
def parse_git_history(history_text, company):
    records = []

    for line in history_text.strip().split("\n"):
        parts = line.split("|", maxsplit=2)

        if len(parts) != 3:
            continue

        commit_hash, timestamp, message = parts

        records.append({
            "company": company,
            "commit_hash": commit_hash.strip(),
            "timestamp": pd.to_datetime(
                timestamp.strip(),
                utc=True
            ),
            "commit_message": message.strip()
        })

    return records

In [46]:
zoom_records = parse_git_history(
    zoom_history,
    "Zoom"
)

zoom_df = pd.DataFrame(zoom_records)

zoom_df

,company,commit_hash,timestamp,commit_message
0,Zoom,2a55363890072a783635c05ff3df6af901fcb1ef,2024-08-05 18:19:53+00:00,Record new changes of Zoom Privacy Policy
1,Zoom,5f21c439d209a348c08bdf631cea3313c72f39ce,2024-07-19 18:19:37+00:00,Record new changes of Zoom Privacy Policy
2,Zoom,5974f03468c0d63146c8a5f9755128a98a9daf71,2024-03-01 06:47:40+00:00,Apply technical or declaration upgrade on Zoom...
3,Zoom,da732810c04d446cefbdeb27d82432307224d6a1,2024-03-17 06:48:06+00:00,Record new changes of Zoom Privacy Policy
4,Zoom,d19d78ea80fa38c2fd34b6028e07f7da5f4e6073,2023-12-20 00:46:10+00:00,Record new changes of Zoom Privacy Policy
5,Zoom,1559d97e5d0b44f00609200e65e1965f0bbb0fb9,2023-12-15 18:28:35+00:00,Record new changes of Zoom Privacy Policy
6,Zoom,da1e3f2173580ff939692c2a75666b1d171c46ea,2023-12-02 11:18:10+00:00,Record new changes of Zoom Privacy Policy
7,Zoom,05d6d4fe3ebb303de242cfcccbc20c9daec2b7cf,2023-09-08 21:57:39+00:00,Record new changes of Zoom Privacy Policy
8,Zoom,1105702377e68c8806ba0eb9a0ae5b6628fde209,2023-09-05 12:18:54+00:00,Record new changes of Zoom Privacy Policy
9,Zoom,87a1e21aab39dba80d29fa06da181c773805ab84,2023-08-11 19:15:46+00:00,Record new changes of Zoom Privacy Policy


In [ ]:
all_records = []

for i, row in privacy_df.iterrows():
    company = row["company"]
    relative_path = Path(company) / "Privacy Policy.md"

    try:
        history = get_git_history(DATASET_PATH, relative_path)

        records = parse_git_history(history, company)
        all_records.extend(records)

    except subprocess.CalledProcessError as e:
        print(f"Error processing {company}: {e}")

print("Total version records:", len(all_records))

In [ ]:
history_df = pd.DataFrame(all_records)

history_df.head()

history_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18336 entries, 0 to 18335
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   company         18336 non-null  str           
 1   commit_hash     18336 non-null  str           
 2   date            18336 non-null  datetime64[us]
 3   commit_message  18336 non-null  str           
dtypes: datetime64[us](1), str(3)
memory usage: 573.1 KB


In [ ]:
history_df = history_df.sort_values(
    by=["company", "date"]
).reset_index(drop=True)

history_df.head(20)

,company,commit_hash,date,commit_message
0,123Greetings,0a37d41432355d380bd486f72ab1258c78987063,2012-10-03,Start tracking 123Greetings Privacy Policy
1,123Greetings,978cb6a43ec8874fb73b77e1b76ac342091c97e3,2012-10-10,Update 123Greetings Privacy Policy
2,123Greetings,a74fa57b4b08ab4867819af11a2c94b62215376a,2018-05-10,Update 123Greetings Privacy Policy
3,123Greetings,f31af92915326aaa76e56d87921edb0fe0c3a6f8,2018-05-19,Update 123Greetings Privacy Policy
4,123Greetings,043f3ea4b20d21cf7472ac7e0fa6faa8a5a4ee95,2018-05-20,Update 123Greetings Privacy Policy
5,123Greetings,f8ee2c1e4b7b44d92a8518a2787eb84d7aa0bab1,2018-05-23,Update 123Greetings Privacy Policy
6,123Greetings,e6d4bad44bf082cc59fb1c27a2ce4e7b26805ed7,2018-05-24,Update 123Greetings Privacy Policy
7,123Greetings,e1d1f75cdb7023c101c71be2f184709c2a8ba454,2020-02-28,Update 123Greetings Privacy Policy
8,123Greetings,f0d46be641f850d6ca47d1e4238b7495e7b47f8e,2020-02-29,Update 123Greetings Privacy Policy
9,123Greetings,0655d9aa69bd53d272ad06afaa70ff5351eb9ce9,2020-03-02,Update 123Greetings Privacy Policy


In [ ]:
summary_df = (
    history_df
    .groupby("company")
    .agg(
        versions=("commit_hash", "count"),
        first_version=("date", "min"),
        last_version=("date", "max")
    )
    .reset_index()
)

summary_df["consecutive_pairs"] = (
    summary_df["versions"] - 1
)

summary_df = summary_df.sort_values(
    by="versions",
    ascending=False
).reset_index(drop=True)

summary_df.head(20)

,company,versions,first_version,last_version,consecutive_pairs
0,Xanga,2484,2012-10-04,2025-09-23,2483
1,Alibaba.com,971,2012-10-04,2026-08-20,970
2,Allstate,922,2012-10-04,2026-08-06,921
3,Windstream,877,2020-11-09,2023-01-14,876
4,Zillow,510,2012-10-04,2025-11-15,509
5,Booking.com,492,2020-10-20,2026-08-20,491
6,Disqus,484,2020-12-08,2026-08-10,483
7,CommentCaMarche,434,2020-12-08,2026-08-28,433
8,Wikimedia Foundation,384,2013-06-20,2026-09-02,383
9,Quora,331,2020-07-29,2025-09-23,330


In [ ]:
print("Companies with Privacy Policy history:", len(summary_df))
print("Total policy versions:", summary_df["versions"].sum())
print("Total possible consecutive version pairs:",
      summary_df["consecutive_pairs"].sum())

print("\nVersion statistics:")
print(summary_df["versions"].describe())

Companies with Privacy Policy history: 307
Total policy versions: 18336
Total possible consecutive version pairs: 18029

Version statistics:
count     307.000000
mean       59.726384
std       182.600308
min         1.000000
25%         6.000000
50%        14.000000
75%        35.000000
max      2484.000000
Name: versions, dtype: float64


In [ ]:
candidates = summary_df[
    summary_df["versions"] >= 10
].copy()

print("Companies with at least 10 versions:", len(candidates))

candidates.head(30)

Companies with at least 10 versions: 194


,company,versions,first_version,last_version,consecutive_pairs
0,Xanga,2484,2012-10-04,2025-09-23,2483
1,Alibaba.com,971,2012-10-04,2026-08-20,970
2,Allstate,922,2012-10-04,2026-08-06,921
3,Windstream,877,2020-11-09,2023-01-14,876
4,Zillow,510,2012-10-04,2025-11-15,509
5,Booking.com,492,2020-10-20,2026-08-20,491
6,Disqus,484,2020-12-08,2026-08-10,483
7,CommentCaMarche,434,2020-12-08,2026-08-28,433
8,Wikimedia Foundation,384,2013-06-20,2026-09-02,383
9,Quora,331,2020-07-29,2025-09-23,330


In [ ]:
medium_candidates = summary_df[
    (summary_df["versions"] >= 5) &
    (summary_df["versions"] < 10)
].copy()

print("Companies with 5–9 versions:", len(medium_candidates))

medium_candidates.head(30)

Companies with 5–9 versions: 58


,company,versions,first_version,last_version,consecutive_pairs
194,Waze,9,2022-02-03,2026-07-14,8
195,Better Stack,9,2022-11-29,2025-11-11,8
196,Substack,9,2025-08-01,2026-06-05,8
197,Steam,9,2023-07-28,2026-08-10,8
198,Scalingo,9,2021-08-16,2026-04-08,8
199,Thunderbird,9,2023-08-11,2026-06-29,8
200,Spark,9,2025-07-21,2026-07-21,8
201,Parler,9,2021-07-20,2022-08-27,8
202,Olvid,9,2021-08-18,2025-09-01,8
203,Lyft,9,2021-10-07,2026-07-01,8


In [ ]:
output_path = PROJECT_ROOT / "data" / "processed" / "policy_history_summary.csv"

summary_df.to_csv(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

Saved to: c:\Users\arjun\Desktop\SneakyToS\data\processed\policy_history_summary.csv


In [ ]:
history_output_path = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "policy_version_history.csv"
)

history_df.to_csv(
    history_output_path,
    index=False
)

print(f"Saved to: {history_output_path}")

Saved to: c:\Users\arjun\Desktop\SneakyToS\data\processed\policy_version_history.csv


In [ ]:
def get_file_at_commit(repo_path, commit_hash, relative_path):
    command = [
        "git",
        "-C",
        str(repo_path),
        "show",
        f"{commit_hash}:{relative_path.as_posix()}"
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=True,
        encoding="utf-8",
        errors="replace"
    )

    return result.stdout

In [ ]:
zoom_oldest = (
    history_df[history_df["company"] == "Zoom"]
    .sort_values("date")
    .iloc[0]
)

zoom_oldest

company                                               Zoom
commit_hash       f5197ca248b099f54512cb518181241af1642749
date                                   2021-08-16 00:00:00
commit_message          Start tracking Zoom Privacy Policy
Name: 17890, dtype: object

In [ ]:
zoom_text = get_file_at_commit(
    DATASET_PATH,
    zoom_oldest["commit_hash"],
    Path("Zoom") / "Privacy Policy.md"
)

print(zoom_text[:3000])

Zoom Privacy Statement

Last updated: June 4, 2021

This statement describes the personal data we collect and/or process (which may include collecting, organizing, structuring, storing, using, or disclosing) to provide products and services offered directly by Zoom Video Communications, Inc. (“Zoom”), including Zoom’s websites and its meetings, webinars, and messaging platform (“Zoom Products” or “Products”). Zoom Products do not include products developed by Zoom that are covered under a separate privacy policy (listed [here](https://zoom.us/docs/en-us/trust/privacy/policies.html)).

What Personal Data Do We Receive?

Personal data is any information from or about an identified or identifiable person, including information that Zoom can associate with an individual person. We may collect, or process on behalf of our customers, the following categories of personal data when you use or interact with Zoom Products:

*   **Account Information**: Information associated with an account that

In [ ]:
import hashlib
def get_text_hash(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

In [ ]:
sample_companies = [
    "Zoom",
    "Airbnb",
    "Facebook",
    "Wikimedia Foundation",
    "Zillow",
    "Alibaba.com"
]

sample_history = history_df[
    history_df["company"].isin(sample_companies)
].copy()

sample_history = sample_history.sort_values(
    ["company", "date"]
).reset_index(drop=True)

sample_history.groupby("company").size()

company
Airbnb                  202
Alibaba.com             971
Facebook                292
Wikimedia Foundation    384
Zillow                  510
Zoom                     30
dtype: int64

In [ ]:
test_companies = [
    "Zoom",
    "Airbnb",
    "Facebook",
    "Wikimedia Foundation"
]

In [ ]:
test_history = history_df[
    history_df["company"].isin(test_companies)
].copy()

test_history = test_history.sort_values(
    ["company", "date"]
).reset_index(drop=True)

print(
    test_history.groupby("company")
    .size()
)

company
Airbnb                  202
Facebook                292
Wikimedia Foundation    384
Zoom                     30
dtype: int64


In [ ]:
version_records = []

for i, row in test_history.iterrows():

    company = row["company"]
    commit_hash = row["commit_hash"]

    relative_path = (
        Path(company) /
        "Privacy Policy.md"
    )

    try:
        text = get_file_at_commit(
            DATASET_PATH,
            commit_hash,
            relative_path
        )

        version_records.append({
            "company": company,
            "commit_hash": commit_hash,
            "date": row["date"],
            "commit_message": row["commit_message"],
            "text": text,
            "text_hash": get_text_hash(text),
            "char_count": len(text),
            "word_count": len(text.split())
        })

        if i % 10 == 0:
            print(
                f"Processed {i + 1}/{len(test_history)}"
            )

    except Exception as e:
        print(
            f"Failed: {company} | {commit_hash}"
        )
        print(e)

Processed 1/908
Processed 11/908
Processed 21/908
Processed 31/908
Processed 41/908
Processed 51/908
Processed 61/908
Processed 71/908
Processed 81/908
Processed 91/908
Processed 101/908
Processed 111/908
Processed 121/908
Processed 131/908
Processed 141/908
Processed 151/908
Processed 161/908
Processed 171/908
Processed 181/908
Processed 191/908
Processed 201/908
Processed 211/908
Processed 221/908
Processed 231/908
Processed 241/908
Processed 251/908
Processed 261/908
Processed 271/908
Processed 281/908
Processed 291/908
Processed 301/908
Processed 311/908
Processed 321/908
Processed 331/908
Processed 341/908
Processed 351/908
Processed 361/908
Processed 371/908
Processed 381/908
Processed 391/908
Processed 401/908
Processed 411/908
Processed 421/908
Processed 431/908
Processed 441/908
Processed 451/908
Processed 461/908
Processed 471/908
Processed 481/908
Processed 491/908
Failed: Wikimedia Foundation | a124d422594f75fea1c3cd2087cd40e38a5157fa
Command '['git', '-C', 'c:\\Users\\arju

In [ ]:
versions_df = pd.DataFrame(
    version_records
)

versions_df.head()

,company,commit_hash,date,commit_message,text,text_hash,char_count,word_count
0,Airbnb,708b27c44a0250f84a16c3976ee811360974266d,2020-10-30,Start tracking Airbnb Privacy Policy,Privacy Policy\n\n\n\n\n====================\n...,d4be481cf22cbdf4fc6a993ed8811d08c7e151c2e898cc...,30252,4383
1,Airbnb,516988a53a0961f4f2d0d3d7ebd26a850cc7faf6,2021-04-26,Update Airbnb Privacy Policy,Privacy Policy\n\n\n\n\n====================\n...,3c9d67ca2fb3d78abd305feed24bd38b143e91b8cd9292...,32412,4648
2,Airbnb,ab04d821179057d9398d806b6e4eed3ec5630733,2021-04-29,Update Airbnb Privacy Policy,Privacy Policy\n\n\n\n\n====================\n...,b417e8f8f74ffb03baf2aeb3fc86d7f0dd86ac9ac55127...,32455,4657
3,Airbnb,b8248ba6a2a3aec1a187bbfd58eb74d729704579,2021-04-29,Update Airbnb Privacy Policy,Privacy Policy\n\n\n\n\n====================\n...,3c9d67ca2fb3d78abd305feed24bd38b143e91b8cd9292...,32412,4648
4,Airbnb,85742eef786b812c6ca6be1d362db3f91cfd9a80,2021-04-29,Update Airbnb Privacy Policy,Privacy Policy\n\n\n\n\n====================\n...,b417e8f8f74ffb03baf2aeb3fc86d7f0dd86ac9ac55127...,32455,4657


In [ ]:
summary_df[
    summary_df["company"] == "Airbnb"
]

history_df[
    history_df["company"] == "Airbnb"
].sort_values("date").head(20)

,company,commit_hash,date,commit_message
635,Airbnb,708b27c44a0250f84a16c3976ee811360974266d,2020-10-30,Start tracking Airbnb Privacy Policy
636,Airbnb,516988a53a0961f4f2d0d3d7ebd26a850cc7faf6,2021-04-26,Update Airbnb Privacy Policy
637,Airbnb,ab04d821179057d9398d806b6e4eed3ec5630733,2021-04-29,Update Airbnb Privacy Policy
638,Airbnb,b8248ba6a2a3aec1a187bbfd58eb74d729704579,2021-04-29,Update Airbnb Privacy Policy
639,Airbnb,85742eef786b812c6ca6be1d362db3f91cfd9a80,2021-04-29,Update Airbnb Privacy Policy
640,Airbnb,e0844344b034291454f5c0e1d25a35e67c63edbe,2021-04-30,Update Airbnb Privacy Policy
653,Airbnb,cb66154e2a12de3598deddfebe6d91c1bb37e696,2021-05-03,Update Airbnb Privacy Policy
652,Airbnb,2a87c6a2556d48298bf2d026a501a4feb7dde44a,2021-05-03,Update Airbnb Privacy Policy
651,Airbnb,a6191311bbc4fada2755a71179719c9ac9ccbb12,2021-05-03,Update Airbnb Privacy Policy
650,Airbnb,a76d0927da699338317f60b16559b4d7f7c02b99,2021-05-03,Update Airbnb Privacy Policy


In [ ]:
duplicate_summary = (
    versions_df
    .groupby("company")
    .agg(
        git_versions=("commit_hash", "count"),
        unique_text_versions=("text_hash", "nunique")
    )
    .reset_index()
)

duplicate_summary["duplicate_versions"] = (
    duplicate_summary["git_versions"]
    -
    duplicate_summary["unique_text_versions"]
)

duplicate_summary

,company,git_versions,unique_text_versions,duplicate_versions
0,Airbnb,202,13,189
1,Facebook,128,54,74


In [ ]:
test_history.groupby("company").size()

company
Airbnb                  202
Facebook                292
Wikimedia Foundation    384
Zoom                     30
dtype: int64

In [ ]:
zoom_history = (
    history_df[history_df["company"] == "Zoom"]
    .sort_values("date")
    .reset_index(drop=True)
)

print(f"Zoom versions to extract: {len(zoom_history)}")

Zoom versions to extract: 30


In [ ]:
zoom_version_records = []

for i, row in zoom_history.iterrows():

    company = row["company"]
    commit_hash = row["commit_hash"]

    relative_path = Path(company) / "Privacy Policy.md"

    try:
        text = get_file_at_commit(
            DATASET_PATH,
            commit_hash,
            relative_path
        )

        zoom_version_records.append({
            "company": company,
            "commit_hash": commit_hash,
            "date": row["date"],
            "commit_message": row["commit_message"],
            "text": text,
            "text_hash": get_text_hash(text),
            "char_count": len(text),
            "word_count": len(text.split())
        })

        print(
            f"[{i + 1}/{len(zoom_history)}] "
            f"{row['date'].date()} | "
            f"{len(text.split())} words"
        )

    except Exception as e:
        print(
            f"FAILED: {company} | {commit_hash}"
        )
        print(e)
        

[1/30] 2021-08-16 | 4679 words
[2/30] 2021-10-07 | 4705 words
[3/30] 2021-11-03 | 4829 words
[4/30] 2021-11-23 | 4829 words
[5/30] 2022-01-13 | 4774 words
[6/30] 2022-04-05 | 4774 words
[7/30] 2022-08-31 | 4775 words
[8/30] 2022-09-16 | 5361 words
[9/30] 2022-09-20 | 5358 words
[10/30] 2022-11-07 | 6080 words
[11/30] 2022-11-08 | 6125 words
[12/30] 2022-11-10 | 6126 words
[13/30] 2022-11-19 | 6126 words
[14/30] 2022-12-30 | 7000 words
[15/30] 2022-12-31 | 6996 words
[16/30] 2023-01-19 | 6996 words
[17/30] 2023-01-19 | 6126 words
[18/30] 2023-02-24 | 7008 words
[19/30] 2023-04-10 | 7143 words
[20/30] 2023-06-30 | 7183 words
[21/30] 2023-08-11 | 7367 words
[22/30] 2023-09-05 | 7369 words
[23/30] 2023-09-08 | 7369 words
[24/30] 2023-12-02 | 7977 words
[25/30] 2023-12-15 | 7946 words
[26/30] 2023-12-20 | 8038 words
[27/30] 2024-03-01 | 8056 words
[28/30] 2024-03-17 | 8056 words
[29/30] 2024-07-19 | 8160 words
[30/30] 2024-08-05 | 8227 words


In [ ]:
zoom_versions_df = pd.DataFrame(zoom_version_records)

zoom_versions_df[
    [
        "date",
        "commit_hash",
        "commit_message",
        "word_count",
        "char_count",
        "text_hash"
    ]
]


,date,commit_hash,commit_message,word_count,char_count,text_hash
0,2021-08-16,f5197ca248b099f54512cb518181241af1642749,Start tracking Zoom Privacy Policy,4679,31458,4b2e66fe8b0ba75a7d2510623c618fba549d2fde360080...
1,2021-10-07,4788e3698c43bc99597d7fd2836b138ef8145e85,Update Zoom Privacy Policy,4705,31710,330b2a9d5e468d54cec04499bcf135d0f34ce223923db0...
2,2021-11-03,4fbd6abb19ff42c3a828c95327248576a4966e0a,Refilter Zoom Privacy Policy,4829,32622,1c3199181727797b4b323da4d20c197d3175d80ac55985...
3,2021-11-23,7669087fd9438400efc97cc939a9a061588e6da2,Update Zoom Privacy Policy,4829,32630,f333569a1fc99346c8b2ebe8ce130aaf06eb8df3dccafd...
4,2022-01-13,476756868644d8f9eeb6746d51418c1964c5ea4c,Update Zoom Privacy Policy,4774,32748,a3470c3441ac72dcfb759bfb8cb547ea92ad2a69b9e7b0...
5,2022-04-05,60927018fbf523614ab99e5cce0c701e75ca657b,Update Zoom Privacy Policy,4774,32748,d392a862224f89866e88e283f8eb50fa1b076af4f4839a...
6,2022-08-31,5d41244ff973b31ad590f8a2c68159be1308ecae,Update Zoom Privacy Policy,4775,32753,193561e5e2492418fcb43cd389aa9c7c73552f6fa98179...
7,2022-09-16,961c91adeeb5859e9e0c99c4467ff17b4b9da348,Update Zoom Privacy Policy,5361,36504,e1818ff782b5e716de749b2dbf91c29a7ea7978105be37...
8,2022-09-20,0c0014d9036d2bed3f4832efb4206a6063c63fc0,Refilter Zoom Privacy Policy,5358,36369,f7fae2bd2f59c7dade098973e6f545b250e0c2faab9972...
9,2022-11-07,3fda78b39fabedb62d68a7957d4a60ec085c89fd,Update Zoom Privacy Policy,6080,40957,86aaa00cc0b4bc81d6310ddc5649f60a281a1ca83d0497...


In [ ]:
duplicate_versions = zoom_versions_df[
    zoom_versions_df.duplicated(
        subset="text_hash",
        keep=False
    )
].sort_values("text_hash")

duplicate_versions[
    [
        "date",
        "commit_message",
        "word_count",
        "text_hash"
    ]
]

,date,commit_message,word_count,text_hash
12,2022-11-19,Update Zoom Privacy Policy,6126,68aebf329cab657d01473e6f1e44b497c0b4da73de6ecd...
16,2023-01-19,Refilter Zoom Privacy Policy,6126,68aebf329cab657d01473e6f1e44b497c0b4da73de6ecd...
14,2022-12-31,Update Zoom Privacy Policy,6996,a7ae050e961dfcd2933d433056a4588cd4c641542fe093...
15,2023-01-19,Update Zoom Privacy Policy,6996,a7ae050e961dfcd2933d433056a4588cd4c641542fe093...


In [ ]:
zoom_output_path = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "zoom_policy_versions.pkl"
)

zoom_versions_df.to_pickle(zoom_output_path)

print(f"Saved to: {zoom_output_path}")


Saved to: c:\Users\arjun\Desktop\SneakyToS\data\processed\zoom_policy_versions.pkl
